# Part 10: MST Insertion

The Merkle Search Tree (MST) is the core data structure of an ATProto repository. It maps record
keys (collection/rkey) to CIDs, organized by key depth. This notebook focuses on insertion: how keys
are placed, how depth is determined, and how the tree shape emerges.

**What you'll learn:**

- MST node structure: keys, values, and subtree pointers
- Key depth from leading zero bits in the hash
- Insertion algorithm: find depth, split if needed, recurse
- How the tree shape depends on key distribution

**Prerequisites:** Part 7 (CID), Part 8 (DAG-CBOR)

**Estimated Time:** 20-25 minutes

## Step 1: Key Depth

In an MST, each key's depth is determined by the number of leading zero bits in its hash. The hash
is typically SHA-256, and the depth is the count of leading zero bits divided by the fanout (usually
8).

For our simplified model, we use a hash function that produces predictable depths for demonstration.

In [ ]:
// Simplified key depth — based on first character of key
// Real implementation: SHA-256 leading zero bits / 8
@interface MSTKeyHelper : NSObject
+ (int)depthForKey:(NSString *)key;
+ (NSString *)hashKey:(NSString *)key;
@end

@implementation MSTKeyHelper
+ (NSString *)hashKey:(NSString *)key {
    // Simplified hash — sum of character values mod 256
    int sum = 0;
    for (int i = 0; i < [key length]; i++) {
        sum = sum + [key characterAtIndex:i];
    }
    return [NSString stringWithFormat:@"%02x", sum % 256];
}
+ (int)depthForKey:(NSString *)key {
    // Depth = number of leading zero bits in hash / 8
    // Simplified: use first char code to determine depth
    int firstChar = [key characterAtIndex:0];
    // Keys starting with 'a' get depth 0, 'b' depth 1, etc.
    if (firstChar >= 'a' && firstChar <= 'z') {
        return (firstChar - 'a') / 5;  // 0-5
    }
    return 0;
}
@end

NSLog(@"app.bsky.feed.post/abc depth: %d", [MSTKeyHelper depthForKey:@"app.bsky.feed.post/abc"]);
NSLog(@"app.bsky.actor.profile/self depth: %d", [MSTKeyHelper depthForKey:@"app.bsky.actor.profile/self"]);
NSLog(@"chat.bsky.convo/xyz depth: %d", [MSTKeyHelper depthForKey:@"chat.bsky.convo/xyz"]);

## Step 2: MST Node

An MST node contains sorted key-value pairs and optional subtree pointers. All keys in a node share
the same depth. Subtree pointers link to nodes at higher depths.

In [ ]:
@interface MSTNode : NSObject
@property (nonatomic, assign) int depth;
@property (nonatomic, strong) NSMutableArray *keys;      // sorted NSString
@property (nonatomic, strong) NSMutableArray *values;    // CID strings
@property (nonatomic, strong) NSMutableArray *subtrees;  // MSTNode pointers
- (void)insertKey:(NSString *)key value:(NSString *)cid;
- (NSString *)getKey:(NSString *)key;
- (int)count;
@end

@implementation MSTNode
- (instancetype)initWithDepth:(int)depth {
    self = [super init];
    if (self) {
        _depth = depth;
        _keys = [NSMutableArray array];
        _values = [NSMutableArray array];
        _subtrees = [NSMutableArray array];
    }
    return self;
}
- (void)insertKey:(NSString *)key value:(NSString *)cid {
    // Insert in sorted order
    int insertAt = [_keys count];
    for (int i = 0; i < [_keys count]; i++) {
        if ([key compare:[_keys objectAtIndex:i]] == NSOrderedAscending) {
            insertAt = i;
            break;
        }
    }
    [_keys insertObject:key atIndex:insertAt];
    [_values insertObject:cid atIndex:insertAt];
}
- (NSString *)getKey:(NSString *)key {
    for (int i = 0; i < [_keys count]; i++) {
        if ([[_keys objectAtIndex:i] isEqualToString:key]) {
            return [_values objectAtIndex:i];
        }
    }
    return nil;
}
- (int)count {
    return [_keys count];
}
@end

MSTNode *root = [[MSTNode alloc] initWithDepth:0];
[root insertKey:@"app.bsky.feed.post/abc" value:@"bafyrei111"];
[root insertKey:@"app.bsky.feed.post/def" value:@"bafyrei222"];
[root insertKey:@"app.bsky.actor.profile/self" value:@"bafyrei333"];

NSLog(@"Root has %d keys", [root count]);
NSLog(@"Post abc: %@", [root getKey:@"app.bsky.feed.post/abc"]);
NSLog(@"Missing: %@", [root getKey:@"app.bsky.feed.post/xyz"]);

## Step 3: MST Insertion Algorithm

Inserting a key into an MST:

1. Compute the key's depth from its hash
2. If the key's depth matches the current node's depth, insert here
3. If the key's depth is greater, find or create the right subtree
4. If the key's depth is less, this key belongs at a lower level

In [ ]:
@interface MST : NSObject
@property (nonatomic, strong) MSTNode *root;
- (void)insertKey:(NSString *)key value:(NSString *)cid;
- (NSString *)getKey:(NSString *)key;
- (int)totalCount;
@end

@implementation MST
- (instancetype)init {
    self = [super init];
    if (self) { _root = [[MSTNode alloc] initWithDepth:0]; }
    return self;
}
- (void)insertKey:(NSString *)key value:(NSString *)cid {
    int depth = [MSTKeyHelper depthForKey:key];
    // Simplified: insert at root if depth matches, otherwise at root
    // Real implementation: walk subtrees to find correct depth level
    if (depth == [_root depth]) {
        [_root insertKey:key value:cid];
    } else if (depth > [_root depth]) {
        // Find or create subtree at this depth
        MSTNode *subtree = nil;
        for (int i = 0; i < [[_root subtrees] count]; i++) {
            MSTNode *s = [[_root subtrees] objectAtIndex:i];
            if (s.depth == depth) { subtree = s; break; }
        }
        if (subtree == nil) {
            subtree = [[MSTNode alloc] initWithDepth:depth];
            [[_root subtrees] addObject:subtree];
        }
        [subtree insertKey:key value:cid];
    }
    NSLog(@"Inserted '%@' at depth %d", key, depth);
}
- (NSString *)getKey:(NSString *)key {
    // Check root first
    NSString *val = [_root getKey:key];
    if (val != nil) return val;
    // Check subtrees
    for (int i = 0; i < [[_root subtrees] count]; i++) {
        NSString *subVal = [[[_root subtrees] objectAtIndex:i] getKey:key];
        if (subVal != nil) return subVal;
    }
    return nil;
}
- (int)totalCount {
    int count = [_root count];
    for (int i = 0; i < [[_root subtrees] count]; i++) {
        count = count + [[[_root subtrees] objectAtIndex:i] count];
    }
    return count;
}
@end

MST *mst = [[MST alloc] init];
[mst insertKey:@"app.bsky.feed.post/abc" value:@"bafyrei111"];
[mst insertKey:@"app.bsky.feed.post/def" value:@"bafyrei222"];
[mst insertKey:@"app.bsky.actor.profile/self" value:@"bafyrei333"];
[mst insertKey:@"chat.bsky.convo/xyz" value:@"bafyrei444"];

NSLog(@"Total keys: %d", [mst totalCount]);
NSLog(@"Post abc: %@", [mst getKey:@"app.bsky.feed.post/abc"]);
NSLog(@"Convo xyz: %@", [mst getKey:@"chat.bsky.convo/xyz"]);

## Step 4: Garazyk Production Anchor

In the Garazyk codebase, `Repository/MST.h` and `Repository/MST.m` implement the full MST:

- `MSTNode` — balanced tree node with keys, values, and subtree CID pointers
- `MST insertRecord:atKey:` — inserts a record, updating the tree path
- `MST deleteRecord:atKey:` — removes a record, merging nodes if needed
- `MST getCID` — computes the root CID from the tree structure
- `MSTPersistence` — serializes/deserializes MST nodes to/from the block store

The production MST uses real SHA-256 for depth calculation and stores each node as a DAG-CBOR block
in the CAR archive.

## Exercise

Add a `deleteKey:` method to `MST` that removes a key from the tree. If the key is in the root node,
remove it. If it's in a subtree, search and remove it there. If a subtree becomes empty after
deletion, remove the subtree pointer.

Hint: add a `removeKey:` method to `MSTNode` first, then use it from `MST`.

In [ ]:
// Exercise: implement deleteKey:
// Your code here...

## Solution

In [ ]:
// Solution: deleteKey removes from node or subtree
@interface MSTNode2 : NSObject
@property (nonatomic, assign) int depth;
@property (nonatomic, strong) NSMutableArray *keys;
@property (nonatomic, strong) NSMutableArray *values;
@property (nonatomic, strong) NSMutableArray *subtrees;
- (void)insertKey:(NSString *)key value:(NSString *)cid;
- (BOOL)removeKey:(NSString *)key;
- (int)count;
@end

@implementation MSTNode2
- (instancetype)initWithDepth:(int)depth {
    self = [super init];
    if (self) {
        _depth = depth;
        _keys = [NSMutableArray array];
        _values = [NSMutableArray array];
        _subtrees = [NSMutableArray array];
    }
    return self;
}
- (void)insertKey:(NSString *)key value:(NSString *)cid {
    int insertAt = [_keys count];
    for (int i = 0; i < [_keys count]; i++) {
        if ([key compare:[_keys objectAtIndex:i]] == NSOrderedAscending) {
            insertAt = i; break;
        }
    }
    [_keys insertObject:key atIndex:insertAt];
    [_values insertObject:cid atIndex:insertAt];
}
- (BOOL)removeKey:(NSString *)key {
    for (int i = 0; i < [_keys count]; i++) {
        if ([[_keys objectAtIndex:i] isEqualToString:key]) {
            [_keys removeObjectAtIndex:i];
            [_values removeObjectAtIndex:i];
            return YES;
        }
    }
    return NO;
}
- (int)count { return [_keys count]; }
@end

@interface MST2 : NSObject
@property (nonatomic, strong) MSTNode2 *root;
- (void)insertKey:(NSString *)key value:(NSString *)cid;
- (BOOL)deleteKey:(NSString *)key;
- (int)totalCount;
@end

@implementation MST2
- (instancetype)init {
    self = [super init];
    if (self) { _root = [[MSTNode2 alloc] initWithDepth:0]; }
    return self;
}
- (void)insertKey:(NSString *)key value:(NSString *)cid {
    [_root insertKey:key value:cid];
}
- (BOOL)deleteKey:(NSString *)key {
    // Try root first
    if ([_root removeKey:key]) return YES;
    // Try subtrees
    for (int i = 0; i < [[_root subtrees] count]; i++) {
        MSTNode2 *sub = [[_root subtrees] objectAtIndex:i];
        if ([sub removeKey:key]) {
            // Remove empty subtree
            if ([sub count] == 0) {
                [[_root subtrees] removeObjectAtIndex:i];
            }
            return YES;
        }
    }
    return NO;
}
- (int)totalCount {
    int count = [_root count];
    for (int i = 0; i < [[_root subtrees] count]; i++) {
        count = count + [[[_root subtrees] objectAtIndex:i] count];
    }
    return count;
}
@end

MST2 *m = [[MST2 alloc] init];
[m insertKey:@"app.bsky.feed.post/abc" value:@"bafyrei111"];
[m insertKey:@"app.bsky.feed.post/def" value:@"bafyrei222"];
NSLog(@"Before delete: %d", [m totalCount]);
[m deleteKey:@"app.bsky.feed.post/abc"];
NSLog(@"After delete: %d", [m totalCount]);
NSLog(@"Delete missing: %d", [m deleteKey:@"nonexistent"]);

## What to Read Next

You now understand MST insertion. Next:

- **Part 11: MST Proofs & Diff** — proof paths, walker, diff operations
- **Part 14: Record Writes** — how records are stored in the MST
- **Part 18: Archaeology — Repository** — production MST implementation